In [8]:
import torch
import torch.nn as nn
from torchvision import models

class GroupedEmotionClassifier(nn.Module):
    def __init__(self, num_classes=6, pretrained=False):
        super(GroupedEmotionClassifier, self).__init__()
        
        resnet = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
        # THIS must match the ORIGINAL architecture that was trained
        # (the one with BatchNorm that saved the .pth file)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),  # This was in original!
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Load model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
emotion_model = GroupedEmotionClassifier(num_classes=6, pretrained=False)
emotion_model.load_state_dict(torch.load('best_model_grouped.pth', map_location=device))
emotion_model = emotion_model.to(device)
emotion_model.eval()

print("✅ Loaded emotion model successfully!")
print(f"   Device: {device}")

# Quick sanity check
test_input = torch.randn(2, 3, 224, 224).to(device)
with torch.no_grad():
    test_output = emotion_model(test_input)
print(f"✅ Forward pass works! Output shape: {test_output.shape}")

✅ Loaded emotion model successfully!
   Device: cpu
✅ Forward pass works! Output shape: torch.Size([2, 6])


In [9]:
from PIL import Image
from torchvision import transforms
import pandas as pd
import torch

# Emotion groups (must match training order)
emotion_groups = ['contempt', 'negative_intense', 'negative_subdued', 
                 'positive_calm', 'positive_energetic', 'surprise']

# Transform
emotion_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_middle_frame(gif_path):
    """Extract middle frame from GIF"""
    try:
        with Image.open(gif_path) as gif:
            n_frames = 0
            try:
                while True:
                    gif.seek(n_frames)
                    n_frames += 1
            except EOFError:
                pass
            
            gif.seek(n_frames // 2)
            return gif.convert('RGB')
    except:
        return None

def detect_emotion(gif_path):
    """Full emotion detection pipeline"""
    # Extract frame
    frame = extract_middle_frame(gif_path)
    if frame is None:
        return None, None, None
    
    # Transform
    frame_tensor = emotion_transform(frame).unsqueeze(0).to(device)
    
    # Predict
    with torch.no_grad():
        output = emotion_model(frame_tensor)
        probabilities = torch.softmax(output, dim=1)
        emotion_idx = torch.argmax(probabilities, dim=1).item()
        confidence = probabilities[0, emotion_idx].item()
    
    # Get all probabilities
    all_probs = {emotion_groups[i]: probabilities[0, i].item() for i in range(6)}
    
    return emotion_groups[emotion_idx], confidence, all_probs

# Test on sample GIFs from each emotion group
test_df = pd.read_csv('test_grouped.csv')

print("=" * 60)
print("🧪 EMOTION DETECTION TEST RESULTS")
print("=" * 60)

# Test 2 samples from each emotion group
correct = 0
total = 0

for emotion in emotion_groups:
    print(f"\n📌 {emotion.upper()}:")
    print("-" * 40)
    
    samples = test_df[test_df['emotion_group'] == emotion].head(3)
    
    for _, row in samples.iterrows():
        gif_path = row['gif_path']
        true_emotion = row['emotion_group']
        
        predicted, confidence, all_probs = detect_emotion(gif_path)
        
        match = "✅" if predicted == true_emotion else "❌"
        total += 1
        if predicted == true_emotion:
            correct += 1
        
        print(f"  {match} ID: {row['gif_id']}")
        print(f"     True: {true_emotion}")
        print(f"     Pred: {predicted} ({confidence:.1%})")
        
        # Show top 3 probabilities
        top3 = sorted(all_probs.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"     Top3: {', '.join([f'{e}:{p:.1%}' for e,p in top3])}")
        print()

print("=" * 60)
print(f"📊 QUICK TEST SUMMARY")
print("=" * 60)
print(f"  Correct: {correct}/{total} ({correct/total*100:.1f}%)")
print(f"  This is a small sample - full test acc is 35.88%")

🧪 EMOTION DETECTION TEST RESULTS

📌 CONTEMPT:
----------------------------------------
  ❌ ID: 4BeeqBqVP6eEU
     True: contempt
     Pred: negative_subdued (39.9%)
     Top3: negative_subdued:39.9%, negative_intense:23.2%, positive_energetic:20.0%

  ❌ ID: 12kUmcu84SN5U4
     True: contempt
     Pred: surprise (96.1%)
     Top3: surprise:96.1%, negative_subdued:1.8%, contempt:1.1%

  ✅ ID: TbRXNJJJbgIkE
     True: contempt
     Pred: contempt (98.7%)
     Top3: contempt:98.7%, negative_subdued:1.3%, negative_intense:0.0%


📌 NEGATIVE_INTENSE:
----------------------------------------
  ❌ ID: b71OApbWu6XL2
     True: negative_intense
     Pred: negative_subdued (70.7%)
     Top3: negative_subdued:70.7%, negative_intense:18.9%, positive_calm:8.8%

  ❌ ID: j5T2a0zWpioBG
     True: negative_intense
     Pred: surprise (88.6%)
     Top3: surprise:88.6%, negative_intense:6.9%, contempt:2.1%

  ❌ ID: 6fl2Fx1eSClmE
     True: negative_intense
     Pred: negative_subdued (98.1%)
     Top3: nega

Even if emotion detection is wrong sometimes:

GIF: Person dancing
Model detects: positive_energetic (correct ✅)
Caption: "A person dancing joyfully" ✅

GIF: Person sitting sadly  
Model detects: positive_energetic (wrong ❌)
Caption: "A person sitting happily" 
→ Still a valid caption! Just wrong emotion tone

The caption system will STILL WORK.
We just need to make sure emotions appear in captions.

In [4]:
%pip install nltk transformers Pillow

  Using cached transformers-5.0.0-py3-none-any.whl.metadata (37 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
     ---------------------------------------- 0.0/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     --------- ------------------------------ 10.2/41.5 kB ? eta -:--:--
     ------------------ ------------------- 20.5/41.5 kB 162.5 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 162.5 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 162.5 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 162.5 kB/s eta 0:00:01
     ------------------ ------------------- 20.5/41.5 kB 162.5 kB/s eta 0:00:01
     -------------------------------------- 41.5/41.5 kB 125.1 kB/s eta 0:00:00
  Using cached huggingface_hub-1.3.5-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached typer_slim-


[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import random
import nltk
nltk.download('punkt')

# Load your test/train data
train_df = pd.read_csv('train_grouped.csv')
val_df = pd.read_csv('val_grouped.csv')
test_df = pd.read_csv('test_grouped.csv')

# ============================================================
# EMOTION VOCABULARY
# ============================================================
emotion_vocabulary = {
    'positive_energetic': {
        'adjectives': ['joyful', 'happy', 'excited', 'cheerful', 'enthusiastic', 'energetic', 'elated'],
        'adverbs': ['happily', 'joyfully', 'excitedly', 'cheerfully', 'enthusiastically'],
        'phrases': [
            'with great joy',
            'in a cheerful manner',
            'full of excitement',
            'with boundless energy',
            'in a joyful way'
        ]
    },
    'positive_calm': {
        'adjectives': ['peaceful', 'content', 'serene', 'relaxed', 'satisfied', 'tranquil'],
        'adverbs': ['peacefully', 'calmly', 'serenely', 'contentedly', 'quietly'],
        'phrases': [
            'in a peaceful manner',
            'with calm contentment',
            'in serene tranquility',
            'with quiet satisfaction',
            'in a relaxed way'
        ]
    },
    'negative_intense': {
        'adjectives': ['angry', 'furious', 'fearful', 'terrified', 'disgusted', 'enraged'],
        'adverbs': ['angrily', 'furiously', 'fearfully', 'terrifiedly', 'disgustedly'],
        'phrases': [
            'with intense anger',
            'in a furious manner',
            'full of fear',
            'with great frustration',
            'in an angry way'
        ]
    },
    'negative_subdued': {
        'adjectives': ['sad', 'sorrowful', 'dejected', 'gloomy', 'melancholic', 'somber'],
        'adverbs': ['sadly', 'sorrowfully', 'dejectedly', 'gloomily', 'quietly'],
        'phrases': [
            'with deep sadness',
            'in a sorrowful manner',
            'feeling dejected',
            'with a heavy heart',
            'in a gloomy way'
        ]
    },
    'surprise': {
        'adjectives': ['surprised', 'shocked', 'astonished', 'amazed', 'stunned', 'bewildered'],
        'adverbs': ['surprisingly', 'shockingly', 'astonishedly', 'amazedly'],
        'phrases': [
            'in a surprised manner',
            'with great astonishment',
            'looking completely shocked',
            'in utter amazement',
            'with wide-eyed surprise'
        ]
    },
    'contempt': {
        'adjectives': ['contemptuous', 'disdainful', 'scornful', 'dismissive', 'snide'],
        'adverbs': ['contemptuously', 'disdainfully', 'scornfully', 'dismissively'],
        'phrases': [
            'with disdain',
            'in a contemptuous manner',
            'showing clear disdain',
            'with scornful disregard',
            'in a dismissive way'
        ]
    }
}

print("✅ Emotion vocabulary defined!")
print(f"\nVocabulary sizes:")
for emotion, vocab in emotion_vocabulary.items():
    print(f"  {emotion:20s}: {len(vocab['adjectives'])} adj, {len(vocab['adverbs'])} adv, {len(vocab['phrases'])} phrases")

✅ Emotion vocabulary defined!

Vocabulary sizes:
  positive_energetic  : 7 adj, 5 adv, 5 phrases
  positive_calm       : 6 adj, 5 adv, 5 phrases
  negative_intense    : 6 adj, 5 adv, 5 phrases
  negative_subdued    : 6 adj, 5 adv, 5 phrases
  surprise            : 6 adj, 4 adv, 5 phrases
  contempt            : 5 adj, 4 adv, 5 phrases


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\akind\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [11]:
class TemplateCaptioner:
    """
    Template-based caption generator
    Guarantees emotion words appear in captions
    This is our SAFETY NET / BASELINE
    """
    
    def __init__(self):
        # Templates with placeholders
        # {adj} = emotion adjective
        # {adv} = emotion adverb  
        # {phrase} = emotion phrase
        
        self.templates = [
            # Subject + action templates
            "A {adj} person {action}",
            "A person {action} {adv}",
            "Someone {action} {phrase}",
            "A {adj} individual {action}",
            "A person {action} with {adj} emotion",
            
            # Scene templates
            "A {adj} scene showing someone {action}",
            "An animated scene of a {adj} person {action}",
            "A {adj} moment captured as someone {action}",
            
            # Emotional description templates
            "A person expressing {adj} feelings while {action}",
            "Someone feeling {adj} and {action}",
            "A {adj} person is shown {action}",
            
            # Action-focused templates
            "The {adj} person {action} enthusiastically",
            "A person {action}, looking {adj}",
            "Feeling {adj}, a person {action}"
        ]
        
        # Common GIF actions
        self.actions = [
            'moving around',
            'doing something',
            'reacting to something',
            'expressing themselves',
            'interacting with others',
            'performing an action',
            'engaging in activity',
            'showing their feelings',
            'reacting emotionally',
            'participating in an event'
        ]
    
    def generate(self, emotion, action=None):
        """Generate a caption given emotion and optional action"""
        
        # Get emotion words
        vocab = emotion_vocabulary[emotion]
        adj = random.choice(vocab['adjectives'])
        adv = random.choice(vocab['adverbs'])
        phrase = random.choice(vocab['phrases'])
        
        # Use provided action or random
        if action is None:
            action = random.choice(self.actions)
        
        # Pick random template
        template = random.choice(self.templates)
        
        # Fill template
        caption = template.format(
            adj=adj,
            adv=adv,
            phrase=phrase,
            action=action
        )
        
        return caption

# Test template captioner
template_captioner = TemplateCaptioner()

print("=" * 60)
print("🧪 TEMPLATE CAPTIONER TEST")
print("=" * 60)

for emotion in emotion_groups:
    print(f"\n📌 {emotion}:")
    for i in range(3):
        caption = template_captioner.generate(emotion)
        print(f"   {i+1}. {caption}")

print(f"\n✅ Template captioner working!")

🧪 TEMPLATE CAPTIONER TEST

📌 contempt:
   1. A scornful individual showing their feelings
   2. An animated scene of a scornful person moving around
   3. The scornful person expressing themselves enthusiastically

📌 negative_intense:
   1. The furious person performing an action enthusiastically
   2. Feeling terrified, a person doing something
   3. A person expressing themselves angrily

📌 negative_subdued:
   1. The sad person reacting to something enthusiastically
   2. Someone feeling dejected and reacting emotionally
   3. A person reacting emotionally gloomily

📌 positive_calm:
   1. A satisfied person is shown engaging in activity
   2. An animated scene of a peaceful person participating in an event
   3. A relaxed individual performing an action

📌 positive_energetic:
   1. An animated scene of a excited person reacting emotionally
   2. The happy person showing their feelings enthusiastically
   3. A excited moment captured as someone performing an action

📌 surprise:
   1.

In [12]:
class EmotionGIFCaptioner:
    """
    Full pipeline: GIF → Emotion Detection → Caption Generation
    Uses template baseline for now (guaranteed emotion in captions)
    """
    
    def __init__(self, emotion_model, device):
        self.emotion_model = emotion_model
        self.device = device
        self.template_captioner = TemplateCaptioner()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def extract_middle_frame(self, gif_path):
        """Extract middle frame"""
        try:
            with Image.open(gif_path) as gif:
                n_frames = 0
                try:
                    while True:
                        gif.seek(n_frames)
                        n_frames += 1
                except EOFError:
                    pass
                gif.seek(n_frames // 2)
                return gif.convert('RGB')
        except:
            return None
    
    def detect_emotion(self, gif_path):
        """Detect emotion from GIF"""
        frame = self.extract_middle_frame(gif_path)
        if frame is None:
            return None, None
        
        frame_tensor = self.transform(frame).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            output = self.emotion_model(frame_tensor)
            probs = torch.softmax(output, dim=1)
            idx = torch.argmax(probs, dim=1).item()
            confidence = probs[0, idx].item()
        
        return emotion_groups[idx], confidence
    
    def generate_caption(self, gif_path, action=None):
        """Full pipeline: GIF → Caption"""
        
        # Step 1: Detect emotion
        emotion, confidence = self.detect_emotion(gif_path)
        
        if emotion is None:
            return "Unable to process GIF", None, None
        
        # Step 2: Generate caption
        caption = self.template_captioner.generate(emotion, action)
        
        return caption, emotion, confidence

# Create captioner
captioner = EmotionGIFCaptioner(emotion_model, device)

print("✅ End-to-end captioner ready!")

✅ End-to-end captioner ready!


In [13]:
# Test on sample GIFs
print("=" * 60)
print("🎬 EMOTION-BASED GIF CAPTIONING - DEMO")
print("=" * 60)

test_df = pd.read_csv('test_grouped.csv')

# Test on 10 random GIFs
samples = test_df.sample(10, random_state=42)

for i, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    true_emotion = row['emotion_group']
    
    # Generate caption
    caption, pred_emotion, confidence = captioner.generate_caption(gif_path)
    
    match = "✅" if pred_emotion == true_emotion else "❌"
    
    print(f"\n{'─' * 60}")
    print(f"  GIF #{i+1}: {row['gif_id']}")
    print(f"  {match} True Emotion:      {true_emotion}")
    print(f"     Detected Emotion:  {pred_emotion} ({confidence:.1%})")
    print(f"  💬 Generated Caption: {caption}")
    print(f"{'─' * 60}")

print(f"\n{'=' * 60}")
print("✅ END-TO-END PIPELINE WORKING!")
print(f"{'=' * 60}")
print("\n🎯 Next Steps:")
print("  1. Add object detection")
print("  2. Add action detection")  
print("  3. Build LSTM decoder for learned captions")
print("  4. Combine everything into final system")

🎬 EMOTION-BASED GIF CAPTIONING - DEMO

────────────────────────────────────────────────────────────
  GIF #1: HSNAwKc63r9lK
  ✅ True Emotion:      negative_intense
     Detected Emotion:  negative_intense (56.1%)
  💬 Generated Caption: An animated scene of a enraged person reacting emotionally
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #2: 9YdllfJmcs5RS
  ✅ True Emotion:      negative_intense
     Detected Emotion:  negative_intense (96.6%)
  💬 Generated Caption: A angry person is shown engaging in activity
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #3: aF8nZCmUEgmBO
  ❌ True Emotion:      surprise
     Detected Emotion:  negative_intense (45.4%)
  💬 Generated Caption: Someone feeling angry and interacting with others
────────────────────────────────────────────────────────────

──────────────────────────────────────

STEP 1: Add Object Detection
We'll use a pre-trained YOLO model (fast, accurate, easy to use). This will detect objects in your GIF frames and give you concrete nouns instead of "something."

In [3]:
%pip install ultralytics pillow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
from ultralytics import YOLO
from PIL import Image
import torch

class ObjectDetector:
    """
    Detects objects in GIF frames using YOLOv8
    Returns top N detected objects with confidence scores
    """
    
    def __init__(self, model_name='yolov8n.pt', confidence_threshold=0.3):
        """
        Args:
            model_name: YOLOv8 model variant ('yolov8n.pt' = nano, fastest)
            confidence_threshold: Minimum confidence to include object
        """
        print(f"Loading YOLO model: {model_name}...")
        self.model = YOLO(model_name)
        self.confidence_threshold = confidence_threshold
        print("✅ YOLO model loaded!")
    
    def detect_objects(self, image_path_or_pil, top_n=3):
        """
        Detect objects in an image
        
        Args:
            image_path_or_pil: Path to image or PIL Image object
            top_n: Return top N most confident objects
        
        Returns:
            List of (object_name, confidence) tuples
        """
        # Run detection
        results = self.model(image_path_or_pil, verbose=False)
        
        # Extract detections
        detections = []
        for result in results:
            boxes = result.boxes
            for box in boxes:
                confidence = float(box.conf[0])
                if confidence >= self.confidence_threshold:
                    class_id = int(box.cls[0])
                    class_name = result.names[class_id]
                    detections.append((class_name, confidence))
        
        # Sort by confidence and take top N
        detections.sort(key=lambda x: x[1], reverse=True)
        return detections[:top_n]
    
    def detect_from_gif(self, gif_path, top_n=3):
        """
        Detect objects from middle frame of GIF
        
        Returns:
            List of object names (just the names, not confidences)
        """
        # Extract middle frame
        try:
            with Image.open(gif_path) as gif:
                n_frames = 0
                try:
                    while True:
                        gif.seek(n_frames)
                        n_frames += 1
                except EOFError:
                    pass
                gif.seek(n_frames // 2)
                frame = gif.convert('RGB')
        except:
            return []
        
        # Detect objects
        detections = self.detect_objects(frame, top_n=top_n)
        
        # Return just the object names
        return [obj for obj, conf in detections]

# Initialize object detector
object_detector = ObjectDetector(model_name='yolov8n.pt', confidence_threshold=0.3)

print("✅ Object detector ready!")

Loading YOLO model: yolov8n.pt...
✅ YOLO model loaded!
✅ Object detector ready!


In [15]:
import pandas as pd

# Load test data
test_df = pd.read_csv('test_grouped.csv')

print("=" * 60)
print("🔍 OBJECT DETECTION TEST")
print("=" * 60)

# Test on 10 sample GIFs
samples = test_df.sample(10, random_state=42)

for i, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    
    # Detect objects
    objects = object_detector.detect_from_gif(gif_path, top_n=3)
    
    print(f"\n📌 GIF #{i+1}: {row['gif_id']}")
    print(f"   Emotion: {row['emotion_group']}")
    print(f"   Objects detected: {', '.join(objects) if objects else 'None'}")

print("\n" + "=" * 60)
print("✅ Object detection working!")
print("=" * 60)

🔍 OBJECT DETECTION TEST

📌 GIF #1: HSNAwKc63r9lK
   Emotion: negative_intense
   Objects detected: person, person

📌 GIF #2: 9YdllfJmcs5RS
   Emotion: negative_intense
   Objects detected: person

📌 GIF #3: aF8nZCmUEgmBO
   Emotion: surprise
   Objects detected: person

📌 GIF #4: HmsAidDdKMTle
   Emotion: positive_energetic
   Objects detected: person, person, person

📌 GIF #5: ZJoCfb9ZMg5q
   Emotion: positive_energetic
   Objects detected: None

📌 GIF #6: TpOm1K4dUV1bW
   Emotion: negative_subdued
   Objects detected: snowboard, airplane

📌 GIF #7: AW8Yqen6SoVmE
   Emotion: positive_energetic
   Objects detected: person, person, person

📌 GIF #8: jhBuX7e7RYwwM
   Emotion: negative_intense
   Objects detected: None

📌 GIF #9: QAgWTCgo9uKI0
   Emotion: positive_energetic
   Objects detected: person

📌 GIF #10: L2Gkc6WuxpgvS
   Emotion: surprise
   Objects detected: person

✅ Object detection working!


In [16]:
class ImprovedTemplateCaptioner:
    """
    Template-based caption generator WITH object detection
    Now captions include actual detected objects!
    """
    
    def __init__(self):
        # Templates with object placeholders
        # {adj} = emotion adjective
        # {adv} = emotion adverb
        # {obj} = detected object
        
        self.templates_with_objects = [
            "A {adj} person with a {obj}",
            "A {adj} {obj} in the scene",
            "A person {adv} interacting with a {obj}",
            "Someone feeling {adj} near a {obj}",
            "A {adj} scene showing a {obj}",
            "A person and a {obj}, both looking {adj}",
            "An animated moment featuring a {adj} {obj}",
        ]
        
        self.templates_no_objects = [
            "A {adj} person moving around",
            "Someone expressing {adj} feelings",
            "A person reacting {adv}",
            "An animated scene of a {adj} person",
            "A {adj} moment captured on camera",
        ]
    
    def generate(self, emotion, objects=None):
        """
        Generate caption with emotion and optional objects
        
        Args:
            emotion: Detected emotion group
            objects: List of detected object names (or None)
        
        Returns:
            Generated caption string
        """
        import random
        
        # Get emotion words
        vocab = emotion_vocabulary[emotion]
        adj = random.choice(vocab['adjectives'])
        adv = random.choice(vocab['adverbs'])
        
        # Choose template based on whether we have objects
        if objects and len(objects) > 0:
            template = random.choice(self.templates_with_objects)
            obj = random.choice(objects)
            caption = template.format(adj=adj, adv=adv, obj=obj)
        else:
            template = random.choice(self.templates_no_objects)
            caption = template.format(adj=adj, adv=adv)
        
        return caption

# Create improved captioner
improved_template_captioner = ImprovedTemplateCaptioner()

print("✅ Improved template captioner ready!")

✅ Improved template captioner ready!


In [17]:
class ImprovedEmotionGIFCaptioner:
    """
    Full pipeline with object detection:
    GIF → Emotion Detection + Object Detection → Caption Generation
    """
    
    def __init__(self, emotion_model, object_detector, device):
        self.emotion_model = emotion_model
        self.object_detector = object_detector
        self.device = device
        self.template_captioner = ImprovedTemplateCaptioner()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def extract_middle_frame(self, gif_path):
        """Extract middle frame"""
        try:
            with Image.open(gif_path) as gif:
                n_frames = 0
                try:
                    while True:
                        gif.seek(n_frames)
                        n_frames += 1
                except EOFError:
                    pass
                gif.seek(n_frames // 2)
                return gif.convert('RGB')
        except:
            return None
    
    def detect_emotion(self, gif_path):
        """Detect emotion from GIF"""
        frame = self.extract_middle_frame(gif_path)
        if frame is None:
            return None, None
        
        frame_tensor = self.transform(frame).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            output = self.emotion_model(frame_tensor)
            probs = torch.softmax(output, dim=1)
            idx = torch.argmax(probs, dim=1).item()
            confidence = probs[0, idx].item()
        
        return emotion_groups[idx], confidence
    
    def generate_caption(self, gif_path):
        """
        Full pipeline: GIF → Caption with emotion + objects
        
        Returns:
            (caption, emotion, confidence, objects)
        """
        
        # Step 1: Detect emotion
        emotion, confidence = self.detect_emotion(gif_path)
        
        if emotion is None:
            return "Unable to process GIF", None, None, []
        
        # Step 2: Detect objects
        objects = self.object_detector.detect_from_gif(gif_path, top_n=3)
        
        # Step 3: Generate caption
        caption = self.template_captioner.generate(emotion, objects)
        
        return caption, emotion, confidence, objects

# Create improved captioner
improved_captioner = ImprovedEmotionGIFCaptioner(emotion_model, object_detector, device)

print("✅ Improved end-to-end captioner ready!")

✅ Improved end-to-end captioner ready!


In [18]:
print("=" * 60)
print("🎬 IMPROVED GIF CAPTIONING - WITH OBJECT DETECTION")
print("=" * 60)

test_df = pd.read_csv('test_grouped.csv')

# Test on 10 random GIFs
samples = test_df.sample(10, random_state=42)

for i, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    true_emotion = row['emotion_group']
    
    # Generate caption
    caption, pred_emotion, confidence, objects = improved_captioner.generate_caption(gif_path)
    
    match = "✅" if pred_emotion == true_emotion else "❌"
    
    print(f"\n{'─' * 60}")
    print(f"  GIF #{i+1}: {row['gif_id']}")
    print(f"  {match} True Emotion:      {true_emotion}")
    print(f"     Detected Emotion:  {pred_emotion} ({confidence:.1%})")
    print(f"  📦 Objects Detected:  {', '.join(objects) if objects else 'None'}")
    print(f"  💬 Generated Caption: {caption}")
    print(f"{'─' * 60}")

print(f"\n{'=' * 60}")
print("✅ OBJECT DETECTION INTEGRATED!")
print(f"{'=' * 60}")
print("\n🎯 Next: Add action detection!")

🎬 IMPROVED GIF CAPTIONING - WITH OBJECT DETECTION

────────────────────────────────────────────────────────────
  GIF #1: HSNAwKc63r9lK
  ✅ True Emotion:      negative_intense
     Detected Emotion:  negative_intense (56.1%)
  📦 Objects Detected:  person, person
  💬 Generated Caption: A terrified person in the scene
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #2: 9YdllfJmcs5RS
  ✅ True Emotion:      negative_intense
     Detected Emotion:  negative_intense (96.6%)
  📦 Objects Detected:  person
  💬 Generated Caption: Someone feeling disgusted near a person
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #3: aF8nZCmUEgmBO
  ❌ True Emotion:      surprise
     Detected Emotion:  negative_intense (45.4%)
  📦 Objects Detected:  person
  💬 Generated Caption: An animated moment featuring a enraged person
──────────────────────────

STEP 2: Add Action Detection
Now we'll detect what's happening in the GIF. This is trickier than object detection because actions require temporal understanding, but we can use a pre-trained action recognition model.
Option A: Simple Action Classification (Recommended for now)
We'll use a pre-trained video classification model from HuggingFace that can recognize common actions.

In [19]:
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification
import torch
import numpy as np
from PIL import Image

class ActionDetector:
    """
    Detects actions in GIFs using VideoMAE
    Returns predicted action label
    """
    
    def __init__(self, model_name="MCG-NJU/videomae-base-finetuned-kinetics"):
        """
        Initialize action detection model
        Using VideoMAE fine-tuned on Kinetics-400 (400 action classes)
        """
        print(f"Loading action detection model: {model_name}...")
        self.processor = VideoMAEImageProcessor.from_pretrained(model_name)
        self.model = VideoMAEForVideoClassification.from_pretrained(model_name)
        self.model.eval()
        print("✅ Action detection model loaded!")
    
    def extract_frames(self, gif_path, num_frames=16):
        """
        Extract evenly spaced frames from GIF
        VideoMAE expects 16 frames as input
        """
        try:
            with Image.open(gif_path) as gif:
                # Count total frames
                n_frames = 0
                try:
                    while True:
                        gif.seek(n_frames)
                        n_frames += 1
                except EOFError:
                    pass
                
                # Extract evenly spaced frames
                if n_frames < num_frames:
                    # If GIF has fewer frames, repeat frames
                    indices = np.linspace(0, n_frames - 1, num_frames, dtype=int)
                else:
                    indices = np.linspace(0, n_frames - 1, num_frames, dtype=int)
                
                frames = []
                for idx in indices:
                    gif.seek(idx)
                    frame = gif.convert('RGB')
                    frames.append(frame)
                
                return frames
        except Exception as e:
            print(f"Error extracting frames: {e}")
            return None
    
    def detect_action(self, gif_path, top_k=3):
        """
        Detect action in GIF
        
        Args:
            gif_path: Path to GIF file
            top_k: Return top K predicted actions
        
        Returns:
            List of (action_name, confidence) tuples
        """
        # Extract frames
        frames = self.extract_frames(gif_path, num_frames=16)
        if frames is None:
            return []
        
        # Preprocess frames
        inputs = self.processor(frames, return_tensors="pt")
        
        # Run inference
        with torch.no_grad():
            outputs = self.model(**inputs)
            logits = outputs.logits
        
        # Get top K predictions
        probs = torch.softmax(logits, dim=-1)[0]
        top_k_probs, top_k_indices = torch.topk(probs, top_k)
        
        # Convert to action names
        results = []
        for prob, idx in zip(top_k_probs, top_k_indices):
            action_name = self.model.config.id2label[idx.item()]
            results.append((action_name, prob.item()))
        
        return results
    
    def get_simple_action(self, gif_path):
        """
        Get single most likely action (simplified)
        Returns just the action name
        """
        actions = self.detect_action(gif_path, top_k=1)
        if actions:
            return actions[0][0]  # Return just the name
        return "moving"  # Default fallback

# Initialize action detector
print("⚠️  WARNING: This will download a ~350MB model on first run")
action_detector = ActionDetector()
print("✅ Action detector ready!")

⚠️  WARNING: This will download a ~350MB model on first run
Loading action detection model: MCG-NJU/videomae-base-finetuned-kinetics...


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

d:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif-env\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\akind\.cache\huggingface\hub\models--MCG-NJU--videomae-base-finetuned-kinetics. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/186 [00:00<?, ?it/s]

✅ Action detection model loaded!
✅ Action detector ready!


In [21]:
import pandas as pd

# Load test data
test_df = pd.read_csv('test_grouped.csv')

print("=" * 60)
print("🎬 ACTION DETECTION TEST")
print("=" * 60)

# Test on 10 sample GIFs
samples = test_df.sample(10, random_state=42)

for i, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    
    # Detect action
    actions = action_detector.detect_action(gif_path, top_k=3)
    
    print(f"\n📌 GIF #{i+1}: {row['gif_id']}")
    print(f"   Emotion: {row['emotion_group']}")
    
    if actions:
        print(f"   Top actions detected:")
        for action, conf in actions:
            print(f"      - {action}: {conf:.1%}")
    else:
        print(f"   No actions detected")

print("\n" + "=" * 60)
print("✅ Action detection working!")
print("=" * 60)

🎬 ACTION DETECTION TEST

📌 GIF #1: HSNAwKc63r9lK
   Emotion: negative_intense
   Top actions detected:
      - air drumming: 20.5%
      - pumping fist: 10.9%
      - headbanging: 9.6%

📌 GIF #2: 9YdllfJmcs5RS
   Emotion: negative_intense
   Top actions detected:
      - singing: 13.0%
      - belly dancing: 5.9%
      - presenting weather forecast: 4.7%

📌 GIF #3: aF8nZCmUEgmBO
   Emotion: surprise
   Top actions detected:
      - rock climbing: 7.3%
      - crying: 7.2%
      - riding unicycle: 3.1%

📌 GIF #4: HmsAidDdKMTle
   Emotion: positive_energetic
   Top actions detected:
      - tap dancing: 56.8%
      - dancing charleston: 18.9%
      - celebrating: 2.5%

📌 GIF #5: ZJoCfb9ZMg5q
   Emotion: positive_energetic
   Top actions detected:
      - playing ukulele: 17.2%
      - playing guitar: 15.2%
      - tapping guitar: 14.9%

📌 GIF #6: TpOm1K4dUV1bW
   Emotion: negative_subdued
   Top actions detected:
      - cleaning gutters: 20.1%
      - cleaning windows: 3.1%
      - kick

In [22]:
class FullTemplateCaptioner:
    """
    Template-based caption generator with:
    - Emotion words
    - Detected objects
    - Detected actions
    """
    
    def __init__(self):
        # Templates with all three components
        self.templates_full = [
            "A {adj} person {action}",
            "Someone {adv} {action}",
            "A {adj} person {action} with a {obj}",
            "A person {action} {adv} near a {obj}",
            "An animated scene of a {adj} person {action}",
            "A {adj} moment showing someone {action}",
        ]
        
        self.templates_no_action = [
            "A {adj} person with a {obj}",
            "A {adj} {obj} in the scene",
            "Someone feeling {adj} near a {obj}",
        ]
        
        self.templates_minimal = [
            "A {adj} person in a scene",
            "An animated moment of a {adj} person",
        ]
    
    def clean_action(self, action):
        """
        Clean up action label from model
        Kinetics labels are like "playing basketball" or "hugging (not baby)"
        """
        # Remove parentheses and content
        import re
        action = re.sub(r'\([^)]*\)', '', action)
        action = action.strip()
        
        # Convert to present continuous if needed
        # "play basketball" → "playing basketball"
        if not action.endswith('ing'):
            # Simple heuristic
            if ' ' in action:
                parts = action.split(' ', 1)
                verb = parts[0]
                if not verb.endswith('ing'):
                    verb = verb + 'ing'
                action = verb + ' ' + parts[1]
        
        return action
    
    def generate(self, emotion, objects=None, action=None):
        """
        Generate caption with emotion, objects, and action
        
        Args:
            emotion: Detected emotion group
            objects: List of detected object names (or None)
            action: Detected action name (or None)
        
        Returns:
            Generated caption string
        """
        import random
        
        # Get emotion words
        vocab = emotion_vocabulary[emotion]
        adj = random.choice(vocab['adjectives'])
        adv = random.choice(vocab['adverbs'])
        
        # Clean action if provided
        if action:
            action = self.clean_action(action)
        
        # Choose template based on what we have
        if action and objects and len(objects) > 0:
            # Full template: emotion + action + object
            template = random.choice(self.templates_full)
            obj = random.choice([o for o in objects if o != 'person']) if any(o != 'person' for o in objects) else 'person'
            caption = template.format(adj=adj, adv=adv, action=action, obj=obj)
        elif action:
            # Action + emotion only
            template = random.choice([t for t in self.templates_full if '{obj}' not in t])
            caption = template.format(adj=adj, adv=adv, action=action)
        elif objects and len(objects) > 0:
            # Object + emotion only
            template = random.choice(self.templates_no_action)
            obj = random.choice([o for o in objects if o != 'person']) if any(o != 'person' for o in objects) else 'person'
            caption = template.format(adj=adj, adv=adv, obj=obj)
        else:
            # Minimal: emotion only
            template = random.choice(self.templates_minimal)
            caption = template.format(adj=adj, adv=adv)
        
        return caption

# Create full captioner
full_template_captioner = FullTemplateCaptioner()

print("✅ Full template captioner ready (emotion + objects + actions)!")

✅ Full template captioner ready (emotion + objects + actions)!


In [23]:
class FullGIFCaptioner:
    """
    Complete pipeline:
    GIF → Emotion + Objects + Action → Caption
    """
    
    def __init__(self, emotion_model, object_detector, action_detector, device):
        self.emotion_model = emotion_model
        self.object_detector = object_detector
        self.action_detector = action_detector
        self.device = device
        self.template_captioner = FullTemplateCaptioner()
        
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def extract_middle_frame(self, gif_path):
        """Extract middle frame for emotion detection"""
        try:
            with Image.open(gif_path) as gif:
                n_frames = 0
                try:
                    while True:
                        gif.seek(n_frames)
                        n_frames += 1
                except EOFError:
                    pass
                gif.seek(n_frames // 2)
                return gif.convert('RGB')
        except:
            return None
    
    def detect_emotion(self, gif_path):
        """Detect emotion from middle frame"""
        frame = self.extract_middle_frame(gif_path)
        if frame is None:
            return None, None
        
        frame_tensor = self.transform(frame).unsqueeze(0).to(self.device)
        
        with torch.no_grad():
            output = self.emotion_model(frame_tensor)
            probs = torch.softmax(output, dim=1)
            idx = torch.argmax(probs, dim=1).item()
            confidence = probs[0, idx].item()
        
        return emotion_groups[idx], confidence
    
    def generate_caption(self, gif_path):
        """
        Full pipeline: GIF → Caption
        
        Returns:
            (caption, emotion, emotion_conf, objects, action, action_conf)
        """
        
        print(f"Processing: {gif_path}")
        
        # Step 1: Detect emotion
        emotion, emotion_conf = self.detect_emotion(gif_path)
        if emotion is None:
            return "Unable to process GIF", None, None, [], None, None
        
        # Step 2: Detect objects
        objects = self.object_detector.detect_from_gif(gif_path, top_n=3)
        
        # Step 3: Detect action
        actions = self.action_detector.detect_action(gif_path, top_k=1)
        action = actions[0][0] if actions else None
        action_conf = actions[0][1] if actions else None
        
        # Step 4: Generate caption
        caption = self.template_captioner.generate(emotion, objects, action)
        
        return caption, emotion, emotion_conf, objects, action, action_conf

# Create full captioner
full_captioner = FullGIFCaptioner(emotion_model, object_detector, action_detector, device)

print("✅ Full GIF captioner ready (emotion + objects + actions)!")

✅ Full GIF captioner ready (emotion + objects + actions)!


In [24]:
print("=" * 60)
print("🎬 COMPLETE GIF CAPTIONING SYSTEM")
print("   Emotion + Objects + Actions")
print("=" * 60)

test_df = pd.read_csv('test_grouped.csv')

# Test on 10 random GIFs
samples = test_df.sample(10, random_state=42)

for i, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    true_emotion = row['emotion_group']
    
    # Generate caption
    caption, pred_emotion, emotion_conf, objects, action, action_conf = full_captioner.generate_caption(gif_path)
    
    match = "✅" if pred_emotion == true_emotion else "❌"
    
    print(f"\n{'─' * 60}")
    print(f"  GIF #{i+1}: {row['gif_id']}")
    print(f"  {match} True Emotion:      {true_emotion}")
    print(f"     Detected Emotion:  {pred_emotion} ({emotion_conf:.1%})")
    print(f"  📦 Objects:           {', '.join(objects) if objects else 'None'}")
    print(f"  🎬 Action:            {action} ({action_conf:.1%})" if action else "  🎬 Action:            None")
    print(f"  💬 Caption:           {caption}")
    print(f"{'─' * 60}")

print(f"\n{'=' * 60}")
print("✅ FULL SYSTEM WORKING!")
print(f"{'=' * 60}")
print("\n🎯 Next: Train LSTM decoder to LEARN captions!")

🎬 COMPLETE GIF CAPTIONING SYSTEM
   Emotion + Objects + Actions
Processing: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images\HSNAwKc63r9lK.gif

────────────────────────────────────────────────────────────
  GIF #1: HSNAwKc63r9lK
  ✅ True Emotion:      negative_intense
     Detected Emotion:  negative_intense (56.1%)
  📦 Objects:           person, person
  🎬 Action:            air drumming (20.5%)
  💬 Caption:           A disgusted person air drumming
────────────────────────────────────────────────────────────
Processing: D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images\9YdllfJmcs5RS.gif

────────────────────────────────────────────────────────────
  GIF #2: 9YdllfJmcs5RS
  ✅ True Emotion:      negative_intense
     Detected Emotion:  negative_intense (96.6%)
  📦 Objects:           person
  🎬 Action:            singing (13.0%)
  💬 Caption:           A terrified person singing with a person
───────────────────────────────────────────